# 🚀 DGX B200 Setup Guide — Amazon ML Challenge

| Info | Details |
|------|----------|
| **URL** | http://172.16.1.90:3118 |
| **Token** | `nvidia@geu2` |
| **Access Period** | 7 Days (March 5–12, 2026) |
| **VPN** | ✅ Already connected |

> **TIP**: Run training overnight! Start today, check tomorrow morning.


---
## 📋 STEP 1 — Check Disk Space & GPU (Run First!)
You need **~100 GB** free:
- Dataset: 5 GB
- Images: 50 GB
- Model cache: 15 GB
- Checkpoints: 20 GB
- Working: 10 GB

In [ ]:
print('=== GPU ===')
!nvidia-smi

print('\n=== Disk Space ===')
!df -h /workspace

print('\n=== RAM ===')
!free -h


---
## 🔑 STEP 2 — Setup Kaggle Credentials

**To get `kaggle.json`:**
1. Go to https://www.kaggle.com
2. Click your profile → Account
3. Scroll to the **API** section
4. Click **Create New API Token**
5. Download `kaggle.json` and upload it here via JupyterLab (↑ icon)

Then run the cell below:

In [ ]:
import os, json

# Write Kaggle credentials directly — no need to upload any file
os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
creds = {'username': 'savetree', 'key': 'KGAT_94c77befcc73c58a23c17e48d77f8acf'}
with open(os.path.expanduser('~/.kaggle/kaggle.json'), 'w') as f:
    json.dump(creds, f)
os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

# Create working directory
os.makedirs('/workspace/multimodal/data', exist_ok=True)

print('✅ Kaggle credentials configured!')


---
## 📦 STEP 3 — Install Dependencies (~15 mins)

In [ ]:
# Step 1: Base packages
!pip install kaggle pandas pillow requests tqdm pyyaml scikit-learn -q

# Step 2: PyTorch with CUDA 12.1
!pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 -q

# Step 3: ML libraries
!pip install transformers==4.45.0 peft==0.12.0 accelerate==0.34.0 bitsandbytes -q
!pip install qwen-vl-utils -q

# Step 4: Flash attention (speeds up training ~2x, takes ~5 min to compile)
!pip install flash-attn --no-build-isolation -q

print('✅ All dependencies installed!')

In [ ]:
# Step 5: Install LLaMA-Factory (training framework)
import os
# Clone only if not already cloned
if not os.path.exists('/workspace/LLaMA-Factory'):
    os.chdir('/workspace')
    !git clone https://github.com/hiyouga/LLaMA-Factory.git

os.chdir('/workspace/LLaMA-Factory')
!pip install -e "." -q
os.makedirs('/workspace/multimodal', exist_ok=True)
os.chdir('/workspace/multimodal')
print('✅ LLaMA-Factory installed!')


---
## 📥 STEP 4 — Download Dataset (~30 mins)

Dataset: https://www.kaggle.com/datasets/suvroo/amazon-ml-challenge

In [ ]:
import os

os.makedirs('/workspace/multimodal/data', exist_ok=True)
os.chdir('/workspace/multimodal/data')

!kaggle datasets download -d suvroo/amazon-ml-challenge --unzip
!ls -lh /workspace/multimodal/data/
print('✅ Dataset downloaded!')


---
## 🖼️ STEP 5 — Download Images (~2–3 hours ⏰)

> It's OK if 5–10% of URLs fail — continue with what you have.

In [ ]:
import pandas as pd, requests, os
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def download_images(csv_path, img_dir):
    os.makedirs(img_dir, exist_ok=True)
    df = pd.read_csv(csv_path)
    print(f'Downloading {len(df)} images to {img_dir}...')

    def dl(row):
        dst = f"{img_dir}/{row['index']}.jpg"
        if os.path.exists(dst): return 'skip'
        try:
            r = requests.get(row['image_link'], timeout=10)
            if r.status_code == 200:
                open(dst, 'wb').write(r.content)
                return 'ok'
        except: pass
        return 'fail'

    ok = skip = fail = 0
    with ThreadPoolExecutor(max_workers=32) as ex:
        futs = {ex.submit(dl, r): r for r in df.to_dict('records')}
        for f in tqdm(as_completed(futs), total=len(futs)):
            s = f.result()
            if s == 'ok': ok += 1
            elif s == 'skip': skip += 1
            else: fail += 1
    print(f'✅ ok={ok} skip={skip} fail={fail}')

# Download train images
download_images('/workspace/multimodal/data/train.csv', '/workspace/multimodal/data/train_images')

# Download test images
download_images('/workspace/multimodal/data/test.csv', '/workspace/multimodal/data/test_images')


---
## 🔧 STEP 6 — Preprocessing (~10 mins)

In [ ]:
import pandas as pd, json, os

# --- 6a: Filter valid training rows ---
ENTITY_UNITS = {
    'item_weight': ['gram','kilogram','microgram','milligram','ounce','pound','ton'],
    'item_volume': ['litre','millilitre','cubic foot','cubic inch','cup','fluid ounce','gallon','pint','quart'],
    'voltage':     ['volt','kilovolt','millivolt'],
    'wattage':     ['watt','kilowatt'],
    'maximum_weight_recommendation': ['gram','kilogram','pound','ton'],
    'height': ['centimetre','foot','inch','metre','millimetre','yard'],
    'width':  ['centimetre','foot','inch','metre','millimetre','yard'],
    'depth':  ['centimetre','foot','inch','metre','millimetre','yard'],
}

def is_valid(val, entity):
    if pd.isna(val) or str(val).strip() == '': return False
    return any(u in str(val).lower() for u in ENTITY_UNITS.get(entity, []))

df = pd.read_csv('/workspace/multimodal/data/train.csv')
print(f'Original: {len(df)} rows')
df_v = df[df.apply(lambda r: is_valid(r.get('entity_value',''), r['entity_name']), axis=1)].copy()
df_v.to_csv('/workspace/multimodal/data/processed_train.csv', index=False)
print(f'After filter: {len(df_v)} rows')

# --- 6b: Create LLaMA-Factory training JSON ---
IMG_DIR = '/workspace/multimodal/data/train_images'
records = []
for _, row in df_v.iterrows():
    img = f"{IMG_DIR}/{row['index']}.jpg"
    if not os.path.exists(img): continue
    records.append({
        'conversations': [
            {'from': 'human', 'value': f"<image>\nWhat is the {row['entity_name']}? Reply ONLY as 'value unit' e.g. '500 gram'."},
            {'from': 'gpt',   'value': str(row['entity_value']).strip()}
        ],
        'images': [img]
    })

out = '/workspace/multimodal/data/for_finetune_created.json'
json.dump(records, open(out, 'w'), indent=2)
print(f'Training samples: {len(records)}')
print(f'Saved: {out}')
print('✅ Preprocessing complete!')


---
## 🏋️ STEP 7 — Training (~4–6 hours ⏰)

**Expected F1 scores:**
| Approach | F1 Score |
|---|---|
| Baseline (no training) | ~0.617 |
| 10k samples | ~0.678 |
| 20k samples | ~0.679 |
| Curated 1600 samples ⭐ | ~0.865 |

**Goal**: F1 > 0.70 (good) · F1 > 0.80 (excellent)

> If OOM errors occur, reduce `per_device_train_batch_size` to 4 and increase `gradient_accumulation_steps` to 16.

In [ ]:
import json, shutil, yaml, os

# --- 7a: Register dataset with LLaMA-Factory ---
shutil.copy('/workspace/multimodal/data/for_finetune_created.json',
            '/workspace/LLaMA-Factory/data/for_finetune_created.json')

info_path = '/workspace/LLaMA-Factory/data/dataset_info.json'
info = json.load(open(info_path))
info['amazon_ml_challenge'] = {
    'file_name': 'for_finetune_created.json',
    'formatting': 'sharegpt',
    'columns': {'messages': 'conversations', 'images': 'images'}
}
json.dump(info, open(info_path, 'w'), indent=2)
print('✅ Dataset registered as: amazon_ml_challenge')

# --- 7b: Create QLora training config ---
cfg = {
    'model_name_or_path': 'Qwen/Qwen2-VL-7B-Instruct',
    'stage': 'sft', 'do_train': True,
    'finetuning_type': 'lora', 'lora_target': 'all',
    'lora_rank': 64, 'lora_alpha': 128, 'lora_dropout': 0.05,
    'quantization_method': 'bitsandbytes', 'quantization_bit': 8,
    'dataset': 'amazon_ml_challenge', 'template': 'qwen2_vl',
    'cutoff_len': 1024, 'overwrite_cache': True, 'preprocessing_num_workers': 4,
    'output_dir': '/workspace/multimodal/saves/qwen2vl_lora_sft',
    'logging_steps': 10, 'save_steps': 100, 'plot_loss': True, 'overwrite_output_dir': True,
    'per_device_train_batch_size': 2, 'gradient_accumulation_steps': 8,
    'learning_rate': 1e-4, 'num_train_epochs': 3,
    'lr_scheduler_type': 'cosine', 'warmup_ratio': 0.1, 'bf16': True,
    'val_size': 0.1, 'per_device_eval_batch_size': 2,
    'eval_strategy': 'steps', 'eval_steps': 100, 'load_best_model_at_end': True
}

path = '/workspace/LLaMA-Factory/examples/train_lora/qwen2vl_amazon.yaml'
os.makedirs(os.path.dirname(path), exist_ok=True)
yaml.dump(cfg, open(path, 'w'), default_flow_style=False)
print(f'✅ Training config saved: {path}')
print('\nConfig summary:')
print(f"  Model   : {cfg['model_name_or_path']}")
print(f"  QLora   : rank={cfg['lora_rank']}, 8-bit quantization")
print(f"  Epochs  : {cfg['num_train_epochs']}")
print(f"  Batch   : {cfg['per_device_train_batch_size']} x {cfg['gradient_accumulation_steps']} grad accum = effective {cfg['per_device_train_batch_size']*cfg['gradient_accumulation_steps']}")


In [ ]:
import os
os.chdir('/workspace/LLaMA-Factory')

print('🏋️ Starting fine-tuning... (4-8 hours)')
print('Training logs will appear below. Do NOT close this tab.\n')

!llamafactory-cli train examples/train_lora/qwen2vl_amazon.yaml


In [ ]:
# Check training results (run after training finishes)
import os

save_dir = '/workspace/multimodal/saves/qwen2vl_lora_sft'
if os.path.exists(save_dir):
    print('✅ Training complete! Files:', os.listdir(save_dir))
    loss_png = os.path.join(save_dir, 'training_loss.png')
    if os.path.exists(loss_png):
        from IPython.display import Image
        display(Image(loss_png))
else:
    print('⏳ Training not complete yet, or output_dir is empty.')


---
## 🔍 STEP 8 — Inference (~1–2 hours ⏰)

In [ ]:
import os, torch, pandas as pd
from PIL import Image
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration
from peft import PeftModel
from tqdm import tqdm

BASE   = 'Qwen/Qwen2-VL-7B-Instruct'
ADAPT  = '/workspace/multimodal/saves/qwen2vl_lora_sft'
IMGS   = '/workspace/multimodal/data/test_images'
OUT    = '/workspace/multimodal/data/test_inf.csv'

PROMPTS = {
    'item_weight': "What is the item weight? Reply ONLY as 'value unit' e.g. '500 gram'.",
    'item_volume': "What is the item volume? Reply ONLY as 'value unit' e.g. '2 litre'.",
    'voltage':     "What is the voltage? Reply ONLY as 'value unit' e.g. '220 volt'.",
    'wattage':     "What is the wattage? Reply ONLY as 'value unit' e.g. '60 watt'.",
    'maximum_weight_recommendation': "What is the maximum weight recommendation? Reply ONLY as 'value unit'.",
    'height': "What is the height? Reply ONLY as 'value unit' e.g. '30 centimetre'.",
    'width':  "What is the width? Reply ONLY as 'value unit'.",
    'depth':  "What is the depth? Reply ONLY as 'value unit'.",
}

print('Loading model...')
model = Qwen2VLForConditionalGeneration.from_pretrained(
    BASE, torch_dtype=torch.float16, device_map='auto', load_in_8bit=True)
model = PeftModel.from_pretrained(model, ADAPT)
model.eval()
proc = AutoProcessor.from_pretrained(BASE)
print('✅ Model loaded')

df = pd.read_csv('/workspace/multimodal/data/test.csv')
results = []

for _, row in tqdm(df.iterrows(), total=len(df), desc='Inference'):
    idx = row['index']
    ip  = f'{IMGS}/{idx}.jpg'
    if not os.path.exists(ip):
        results.append({'index': idx, 'prediction': ''}); continue
    try:
        img    = Image.open(ip).convert('RGB')
        prompt = PROMPTS.get(row['entity_name'], f"What is the {row['entity_name']}? Reply as 'value unit'.")
        msgs   = [{'role':'user','content':[{'type':'image','image':img},{'type':'text','text':prompt}]}]
        txt    = proc.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        inp    = proc(text=[txt], images=[img], return_tensors='pt').to(model.device)
        with torch.no_grad():
            out = model.generate(**inp, max_new_tokens=64, do_sample=False)
        gen  = [o[len(i):] for i, o in zip(inp.input_ids, out)]
        pred = proc.batch_decode(gen, skip_special_tokens=True)[0].strip()
        results.append({'index': idx, 'prediction': pred})
    except:
        results.append({'index': idx, 'prediction': ''})

pd.DataFrame(results).to_csv(OUT, index=False)
print(f'✅ Saved {len(results)} predictions to {OUT}')


---
## 🧹 STEP 9 — Post-processing (~2 mins)

In [ ]:
import re, pandas as pd

UNITS = {
    'item_weight': ['gram','kilogram','microgram','milligram','ounce','pound','ton'],
    'item_volume': ['litre','millilitre','cubic foot','cubic inch','cup','fluid ounce','gallon','pint','quart'],
    'voltage':     ['volt','kilovolt','millivolt'],
    'wattage':     ['watt','kilowatt'],
    'maximum_weight_recommendation': ['gram','kilogram','pound','ton'],
    'height': ['centimetre','foot','inch','metre','millimetre','yard'],
    'width':  ['centimetre','foot','inch','metre','millimetre','yard'],
    'depth':  ['centimetre','foot','inch','metre','millimetre','yard'],
}

def validate(pred, entity):
    pred = str(pred).strip().lower()
    m = re.search(r'([\d\.]+)\s*([a-z ]+)', pred)
    if not m: return ''
    num, unit = m.group(1).strip(), m.group(2).strip()
    for a in UNITS.get(entity, []):
        if a in unit or unit in a:
            return f'{num} {a}'
    return ''

test = pd.read_csv('/workspace/multimodal/data/test.csv')
raw  = pd.read_csv('/workspace/multimodal/data/test_inf.csv')
df   = test.merge(raw, on='index', how='left').fillna('')
df['prediction'] = [validate(r['prediction'], r['entity_name']) for _, r in df.iterrows()]

sub_path = '/workspace/multimodal/data/submission.csv'
df[['index','prediction']].to_csv(sub_path, index=False)

sub = df[['index','prediction']]
print(f'✅ submission.csv saved!')
print(f'   Total rows : {len(sub)}')
print(f'   Non-empty  : {(sub.prediction != "").sum()}')
print(f'   Empty      : {(sub.prediction == "").sum()}')
print()
print(sub.head(10).to_string())


---
## 📤 STEP 10 — Submit to Kaggle

In [ ]:
import os, pandas as pd

sub_path = '/workspace/multimodal/data/submission.csv'
if os.path.exists(sub_path):
    sub = pd.read_csv(sub_path)
    print(f'✅ submission.csv ready: {len(sub)} rows')
    print(f'   Non-empty: {(sub.prediction != "").sum()}')
    print()
    print('TO SUBMIT:')
    print('1. In JupyterLab left panel → navigate to /workspace/multimodal/data/')
    print('2. Right-click submission.csv → Download')
    print('3. Go to https://www.kaggle.com/competitions/amazon-ml-challenge')
    print('4. Click Submit Predictions → upload submission.csv')
else:
    print('❌ submission.csv not found. Run post-processing step first.')


---
## 🗑️ OPTIONAL — Free Up Disk Space

In [ ]:
# Run ONLY after training is complete and submission.csv is downloaded!

# Uncomment to free space:
# !rm -rf ~/.cache/huggingface/hub/
# !rm -rf /workspace/multimodal/data/train_images/

!df -h /workspace
print('⚠️  Uncomment lines above ONLY after downloading your submission.csv!')


---
## 📧 Contacts & Acknowledgment

**Technical Issues:**
- Email: nvidiaaihpc@geu.ac.in
- WhatsApp: 9897755009 · 9548283808

---

### 🏆 Required Acknowledgment

> *"This research utilized the NVIDIA DGX B200 system provided by the NVIDIA Centre for AI and HPC, Graphic Era (Deemed to be) University."*

---
**Good luck! 🚀**